In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# ==============================================================================
# 0. DATEN LADEN & ZIELVARIABLE DEFINIEREN
# ==============================================================================
try:
    df_pipeline = df_model_neighbourhood_ohe.copy()
except NameError:
    df_pipeline = pd.read_pickle("df_model_neighbourhood_ohe.pkl")

# Premium-Listing via Top-25%-Quantil
threshold_top_25 = df_pipeline["review_scores_rating"].quantile(0.75)
df_pipeline["is_premium_listing"] = (
    df_pipeline["review_scores_rating"] >= threshold_top_25
).astype(int)

# Spalten bereinigen (Leakage und Zielvariable entfernen)
base_drop_cols = ["review_scores_rating", "is_premium_listing"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_pipeline.columns:
        base_drop_cols.append(c)

# Modellparameter (reduzierte Modellkapazität gegen Overfitting bei kleineren N)
seg_params = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
}

# ==============================================================================
# 1. GLOBALES GESAMTMARKT-MODELL (BASELINE)
# ==============================================================================
y_global = df_pipeline["is_premium_listing"]
X_global = df_pipeline.drop(columns=base_drop_cols)

X_g_temp, X_g_test, y_g_temp, y_g_test = train_test_split(
    X_global, y_global, test_size=0.20, random_state=42, stratify=y_global
)
X_g_train, X_g_val, y_g_train, y_g_val = train_test_split(
    X_g_temp, y_g_temp, test_size=0.25, random_state=42, stratify=y_g_temp
)

xgb_global = XGBClassifier(**seg_params)
xgb_global.fit(X_g_train, y_g_train, eval_set=[(X_g_val, y_g_val)], verbose=False)

g_tr_acc = accuracy_score(y_g_train, xgb_global.predict(X_g_train))
g_te_acc = accuracy_score(y_g_test, xgb_global.predict(X_g_test))
g_te_auc = roc_auc_score(y_g_test, xgb_global.predict_proba(X_g_test)[:, 1])

results = [
    {
        "Modell / Segment": "Gesamtmarkt",
        "Stichprobe (N)": len(df_pipeline),
        "Train Accuracy": g_tr_acc,
        "Test Accuracy": g_te_acc,
        "Δ Test Acc": 0.0,
        "Test ROC-AUC": g_te_auc,
        "Δ Test ROC-AUC": 0.0,
    }
]

# ==============================================================================
# 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE)
# ==============================================================================
# Identifikation der relevanten One-Hot-Spalten für room_type
room_dummy_cols = [c for c in df_pipeline.columns if c.startswith("room_type_")]

for col in room_dummy_cols:
    seg_name = col.replace("room_type_", "")
    df_sub = df_pipeline[df_pipeline[col] == 1].copy()

    if len(df_sub) < 150:
        continue

    # Alle Zimmertyp-Dummies aus dem Sub-Feature-Set entfernen
    drop_cols_sub = base_drop_cols + room_dummy_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["is_premium_listing"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42, stratify=y_sub
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
    )

    xgb_sub = XGBClassifier(**seg_params)
    xgb_sub.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    tr_acc = accuracy_score(y_train, xgb_sub.predict(X_train))
    te_acc = accuracy_score(y_test, xgb_sub.predict(X_test))
    te_auc = roc_auc_score(y_test, xgb_sub.predict_proba(X_test)[:, 1])

    results.append(
        {
            "Modell / Segment": f"Zimmertyp: {seg_name}",
            "Stichprobe (N)": len(df_sub),
            "Train Accuracy": tr_acc,
            "Test Accuracy": te_acc,
            "Δ Test Acc": te_acc - g_te_acc,
            "Test ROC-AUC": te_auc,
            "Δ Test ROC-AUC": te_auc - g_te_auc,
        }
    )

# ==============================================================================
# 3. SEGMENTIERUNG NACH REISE-MOTIVATION (KURZZEIT VS. LANGZEIT)
# ==============================================================================
df_pipeline["is_langzeit"] = (df_pipeline["minimum_nights"] >= 30).astype(int)

time_segments = {
    "Kurzzeit-Tourismus (< 30 Nächte)": df_pipeline["is_langzeit"] == 0,
    "Langzeit-Wohnen (≥ 30 Nächte)": df_pipeline["is_langzeit"] == 1,
}

for seg_name, condition in time_segments.items():
    df_sub = df_pipeline[condition].copy()

    if len(df_sub) < 100:
        continue

    # Zeitvariablen entfernen, die die Aufteilung definieren
    time_drop_cols = [
        "minimum_nights",
        "min_nights_to_availability_ratio",
        "is_langzeit",
    ]
    drop_cols_sub = base_drop_cols + time_drop_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["is_premium_listing"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42, stratify=y_sub
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
    )

    xgb_sub = XGBClassifier(**seg_params)
    xgb_sub.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    tr_acc = accuracy_score(y_train, xgb_sub.predict(X_train))
    te_acc = accuracy_score(y_test, xgb_sub.predict(X_test))
    te_auc = roc_auc_score(y_test, xgb_sub.predict_proba(X_test)[:, 1])

    results.append(
        {
            "Modell / Segment": f"Reise-Motivation: {seg_name}",
            "Stichprobe (N)": len(df_sub),
            "Train Accuracy": tr_acc,
            "Test Accuracy": te_acc,
            "Δ Test Acc": te_acc - g_te_acc,
            "Test ROC-AUC": te_auc,
            "Δ Test ROC-AUC": te_auc - g_te_auc,
        }
    )

# ==============================================================================
# 4. TABELLARISCHER ERGEBNISVERGLEICH FÜR DIE MASTERARBEIT
# ==============================================================================
df_results = pd.DataFrame(results).set_index("Modell / Segment")

print("\n" + "=" * 95)
print("=== VERGLEICH DER KLASSIFIKATIONSGÜTE: GESAMTMARKT VS. SEGMENTSPEZIFISCHE MODELLE ===")
print("=" * 95)
print(
    df_results.to_string(
        formatters={
            "Stichprobe (N)": "{:,}".format,
            "Train Accuracy": "{:.4f}".format,
            "Test Accuracy": "{:.4f}".format,
            "Δ Test Acc": "{:+.4f}".format,
            "Test ROC-AUC": "{:.4f}".format,
            "Δ Test ROC-AUC": "{:+.4f}".format,
        }
    )
)
print("=" * 95)


=== VERGLEICH DER KLASSIFIKATIONSGÜTE: GESAMTMARKT VS. SEGMENTSPEZIFISCHE MODELLE ===
                                                   Stichprobe (N) Train Accuracy Test Accuracy Δ Test Acc Test ROC-AUC Δ Test ROC-AUC
Modell / Segment                                                                                                                     
Gesamtmarkt                                                 6,298         0.8129        0.7460    +0.0000       0.7695        +0.0000
Zimmertyp: Entire home/apt                                  4,546         0.8309        0.7615    +0.0155       0.7822        +0.0127
Zimmertyp: Private room                                     1,594         0.8326        0.6959    -0.0501       0.7593        -0.0101
Reise-Motivation: Kurzzeit-Tourismus (< 30 Nächte)          5,675         0.8059        0.7683    +0.0223       0.7911        +0.0217
Reise-Motivation: Langzeit-Wohnen (≥ 30 Nächte)               623         0.9437        0.6560    -0.0900    

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


def calculate_adjusted_r2(r2, n, p):
    if n - p - 1 <= 0:
        return np.nan
    return 1 - ((1 - r2) * (n - 1) / (n - p - 1))


# 0. Daten laden
try:
    df_pipeline = df_model_base_no_geo.copy()
except NameError:
    df_pipeline = pd.read_pickle("df_model_base_no_geo.pkl")

# Bereinigung
base_drop_cols = ["review_scores_rating"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_pipeline.columns:
        base_drop_cols.append(c)

xgb_params = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
}

print(
    "--- STARTE FINALE EVALUIERUNG: BASELINE VS. SEGMENTIERUNGEN (INKL. ADJUSTIERTEM R²) ---\n"
)

# ==============================================================================
# 1. GLOBALE BASELINE (GESAMTMARKT)
# ==============================================================================
y_global = df_pipeline["review_scores_rating"]
X_global = df_pipeline.drop(columns=base_drop_cols)

X_g_temp, X_g_test, y_g_temp, y_g_test = train_test_split(
    X_global, y_global, test_size=0.20, random_state=42
)
X_g_train, X_g_val, y_g_train, y_g_val = train_test_split(
    X_g_temp, y_g_temp, test_size=0.25, random_state=42
)

xgb_global = XGBRegressor(**xgb_params)
xgb_global.fit(X_g_train, y_g_train, eval_set=[(X_g_val, y_g_val)], verbose=False)

pred_g_train = xgb_global.predict(X_g_train)
pred_g_test = xgb_global.predict(X_g_test)

g_tr_r2 = r2_score(y_g_train, pred_g_train)
g_te_r2 = r2_score(y_g_test, pred_g_test)
g_tr_mae = mean_absolute_error(y_g_train, pred_g_train)
g_te_mae = mean_absolute_error(y_g_test, pred_g_test)

g_tr_adj_r2 = calculate_adjusted_r2(g_tr_r2, X_g_train.shape[0], X_g_train.shape[1])
g_te_adj_r2 = calculate_adjusted_r2(g_te_r2, X_g_test.shape[0], X_g_test.shape[1])

print("=" * 88)
print("=== GLOBALE BASELINE ERGEBNISSE (GESAMTMARKT) ===")
print(f"Datensätze Gesamtmarkt (N): {len(df_pipeline)}")
print("-" * 88)
print(f"Global Train R²:           {g_tr_r2:.4f}")
print(f"Global Train Adjusted R²:  {g_tr_adj_r2:.4f}")
print(f"Global Train MAE:          {g_tr_mae:.4f}")
print("-" * 88)
print(f"Global Test R²:            {g_te_r2:.4f}")
print(f"Global Test Adjusted R²:   {g_te_adj_r2:.4f}")
print(f"Global Test MAE:           {g_te_mae:.4f}")
print("=" * 88 + "\n\n")

# ==============================================================================
# 2. SEGMENTIERUNG NACH ZIMMERTYP
# ==============================================================================
print("=" * 88)
print("=== 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE) ===")
print("=" * 88)

room_dummy_cols = [c for c in df_pipeline.columns if c.startswith("room_type_")]

for col in room_dummy_cols:
    seg_name = col.replace("room_type_", "")
    df_sub = df_pipeline[df_pipeline[col] == 1].copy()

    if len(df_sub) < 150:
        continue

    drop_cols_sub = base_drop_cols + room_dummy_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["review_scores_rating"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    xgb_seg = XGBRegressor(**xgb_params)
    xgb_seg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    pred_tr = xgb_seg.predict(X_train)
    pred_te = xgb_seg.predict(X_test)

    tr_r2 = r2_score(y_train, pred_tr)
    te_r2 = r2_score(y_test, pred_te)
    tr_mae = mean_absolute_error(y_train, pred_tr)
    te_mae = mean_absolute_error(y_test, pred_te)

    tr_adj_r2 = calculate_adjusted_r2(tr_r2, X_train.shape[0], X_train.shape[1])
    te_adj_r2 = calculate_adjusted_r2(te_r2, X_test.shape[0], X_test.shape[1])

    print(f"\n▶ SEGMENT: {seg_name} (N = {len(df_sub)})")
    print("-" * 60)
    print(f"Train R²:          {tr_r2:.4f}")
    print(f"Train Adjusted R²: {tr_adj_r2:.4f}")
    print(f"Train MAE:         {tr_mae:.4f}")
    print("-" * 60)
    print(
        f"Test R²:           {te_r2:.4f}  (Delta zu Global Baseline: {te_r2 - g_te_r2:+.4f})"
    )
    print(
        f"Test Adjusted R²:  {te_adj_r2:.4f}  (Delta zu Global Baseline: {te_adj_r2 - g_te_adj_r2:+.4f})"
    )
    print(
        f"Test MAE:          {te_mae:.4f}  (Delta zu Global Baseline: {te_mae - g_te_mae:+.4f})"
    )
    print("#" * 88)

# ==============================================================================
# 3. SEGMENTIERUNG NACH REISE-MOTIVATION
# ==============================================================================
print("\n\n" + "=" * 88)
print("=== 3. SEGMENTIERUNG NACH REISE-MOTIVATION (ZEIT-HORIZONT) ===")
print("=" * 88)

df_pipeline["time_segment"] = np.where(
    df_pipeline["minimum_nights"] < 30,
    "1_KURZZEIT_TOURISMUS (< 30 Nächte)",
    "2_LANGZEIT_WOHNEN (>= 30 Nächte)",
)

zeit_segmente = sorted(df_pipeline["time_segment"].unique())

for seg_name in zeit_segmente:
    df_sub = df_pipeline[df_pipeline["time_segment"] == seg_name].copy()

    if len(df_sub) < 100:
        continue

    time_drop_cols = [
        "minimum_nights",
        "min_nights_to_availability_ratio",
        "time_segment",
    ]
    drop_cols_sub = base_drop_cols + time_drop_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["review_scores_rating"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    xgb_seg = XGBRegressor(**xgb_params)
    xgb_seg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    pred_tr = xgb_seg.predict(X_train)
    pred_te = xgb_seg.predict(X_test)

    tr_r2 = r2_score(y_train, pred_tr)
    te_r2 = r2_score(y_test, pred_te)
    tr_mae = mean_absolute_error(y_train, pred_tr)
    te_mae = mean_absolute_error(y_test, pred_te)

    tr_adj_r2 = calculate_adjusted_r2(tr_r2, X_train.shape[0], X_train.shape[1])
    te_adj_r2 = calculate_adjusted_r2(te_r2, X_test.shape[0], X_test.shape[1])

    print(f"\n▶ SEGMENT: {seg_name} (N = {len(df_sub)})")
    print("-" * 60)
    print(f"Train R²:          {tr_r2:.4f}")
    print(f"Train Adjusted R²: {tr_adj_r2:.4f}")
    print(f"Train MAE:         {tr_mae:.4f}")
    print("-" * 60)
    print(
        f"Test R²:           {te_r2:.4f}  (Delta zu Global Baseline: {te_r2 - g_te_r2:+.4f})"
    )
    print(
        f"Test Adjusted R²:  {te_adj_r2:.4f}  (Delta zu Global Baseline: {te_adj_r2 - g_te_adj_r2:+.4f})"
    )
    print(
        f"Test MAE:          {te_mae:.4f}  (Delta zu Global Baseline: {te_mae - g_te_mae:+.4f})"
    )
    print("#" * 88)

--- STARTE FINALE EVALUIERUNG: BASELINE VS. SEGMENTIERUNGEN (INKL. ADJUSTIERTEM R²) ---

=== GLOBALE BASELINE ERGEBNISSE (GESAMTMARKT) ===
Datensätze Gesamtmarkt (N): 6298
----------------------------------------------------------------------------------------
Global Train R²:           0.5160
Global Train Adjusted R²:  0.5123
Global Train MAE:          0.1214
----------------------------------------------------------------------------------------
Global Test R²:            0.2660
Global Test Adjusted R²:   0.2487
Global Test MAE:           0.1438


=== 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE) ===

▶ SEGMENT: Entire home/apt (N = 4546)
------------------------------------------------------------
Train R²:          0.5421
Train Adjusted R²: 0.5379
Train MAE:         0.1167
------------------------------------------------------------
Test R²:           0.3636  (Delta zu Global Baseline: +0.0976)
Test Adjusted R²:  0.3456  (Delta zu Global Baseline: +0.0969)
Test MAE:          0.1391  

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


def calculate_adjusted_r2(r2, n, p):
    if n - p - 1 <= 0:
        return np.nan
    return 1 - ((1 - r2) * (n - 1) / (n - p - 1))


# 0. Daten laden
try:
    df_pipeline = df_model_neighbourhood_ohe.copy()
except NameError:
    df_pipeline = pd.read_pickle("df_model_neighbourhood_ohe.pkl")

# Bereinigung
base_drop_cols = ["review_scores_rating"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_pipeline.columns:
        base_drop_cols.append(c)

xgb_params = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
}

print(
    "--- STARTE FINALE EVALUIERUNG: OHE-NACHBARSCHAFTEN (INKL. ADJUSTIERTEM R²) ---\n"
)

# 1. GLOBALE BASELINE
y_global = df_pipeline["review_scores_rating"]
X_global = df_pipeline.drop(columns=base_drop_cols)

X_g_temp, X_g_test, y_g_temp, y_g_test = train_test_split(
    X_global, y_global, test_size=0.20, random_state=42
)
X_g_train, X_g_val, y_g_train, y_g_val = train_test_split(
    X_g_temp, y_g_temp, test_size=0.25, random_state=42
)

xgb_global = XGBRegressor(**xgb_params)
xgb_global.fit(X_g_train, y_g_train, eval_set=[(X_g_val, y_g_val)], verbose=False)

pred_g_train = xgb_global.predict(X_g_train)
pred_g_test = xgb_global.predict(X_g_test)

g_tr_r2 = r2_score(y_g_train, pred_g_train)
g_te_r2 = r2_score(y_g_test, pred_g_test)
g_tr_mae = mean_absolute_error(y_g_train, pred_g_train)
g_te_mae = mean_absolute_error(y_g_test, pred_g_test)

g_tr_adj_r2 = calculate_adjusted_r2(g_tr_r2, X_g_train.shape[0], X_g_train.shape[1])
g_te_adj_r2 = calculate_adjusted_r2(g_te_r2, X_g_test.shape[0], X_g_test.shape[1])

print("=" * 88)
print("=== GLOBALE BASELINE ERGEBNISSE (GESAMTMARKT MIT OHE-KIEZEN) ===")
print(f"Datensätze Gesamtmarkt (N): {len(df_pipeline)}")
print("-" * 88)
print(f"Global Train R²:           {g_tr_r2:.4f}")
print(f"Global Train Adjusted R²:  {g_tr_adj_r2:.4f}")
print(f"Global Train MAE:          {g_tr_mae:.4f}")
print("-" * 88)
print(f"Global Test R²:            {g_te_r2:.4f}")
print(f"Global Test Adjusted R²:   {g_te_adj_r2:.4f}")
print(f"Global Test MAE:           {g_te_mae:.4f}")
print("=" * 88 + "\n\n")

# 2. SEGMENTIERUNG NACH ZIMMERTYP
print("=" * 88)
print("=== 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE) ===")
print("=" * 88)

room_dummy_cols = [c for c in df_pipeline.columns if c.startswith("room_type_")]

for col in room_dummy_cols:
    seg_name = col.replace("room_type_", "")
    df_sub = df_pipeline[df_pipeline[col] == 1].copy()

    if len(df_sub) < 150:
        continue

    drop_cols_sub = base_drop_cols + room_dummy_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["review_scores_rating"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    xgb_seg = XGBRegressor(**xgb_params)
    xgb_seg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    pred_tr = xgb_seg.predict(X_train)
    pred_te = xgb_seg.predict(X_test)

    tr_r2 = r2_score(y_train, pred_tr)
    te_r2 = r2_score(y_test, pred_te)
    tr_mae = mean_absolute_error(y_train, pred_tr)
    te_mae = mean_absolute_error(y_test, pred_te)

    tr_adj_r2 = calculate_adjusted_r2(tr_r2, X_train.shape[0], X_train.shape[1])
    te_adj_r2 = calculate_adjusted_r2(te_r2, X_test.shape[0], X_test.shape[1])

    print(f"\n▶ SEGMENT: {seg_name} (N = {len(df_sub)})")
    print("-" * 60)
    print(f"Train R²:          {tr_r2:.4f}")
    print(f"Train Adjusted R²: {tr_adj_r2:.4f}")
    print(f"Train MAE:         {tr_mae:.4f}")
    print("-" * 60)
    print(
        f"Test R²:           {te_r2:.4f}  (Delta zu Global Baseline: {te_r2 - g_te_r2:+.4f})"
    )
    print(
        f"Test Adjusted R²:  {te_adj_r2:.4f}  (Delta zu Global Baseline: {te_adj_r2 - g_te_adj_r2:+.4f})"
    )
    print(
        f"Test MAE:          {te_mae:.4f}  (Delta zu Global Baseline: {te_mae - g_te_mae:+.4f})"
    )
    print("#" * 88)

# 3. SEGMENTIERUNG NACH REISE-MOTIVATION
print("\n\n" + "=" * 88)
print("=== 3. SEGMENTIERUNG NACH REISE-MOTIVATION (ZEIT-HORIZONT) ===")
print("=" * 88)

df_pipeline["time_segment"] = np.where(
    df_pipeline["minimum_nights"] < 30,
    "1_KURZZEIT_TOURISMUS (< 30 Nächte)",
    "2_LANGZEIT_WOHNEN (>= 30 Nächte)",
)

zeit_segmente = sorted(df_pipeline["time_segment"].unique())

for seg_name in zeit_segmente:
    df_sub = df_pipeline[df_pipeline["time_segment"] == seg_name].copy()

    if len(df_sub) < 100:
        continue

    time_drop_cols = [
        "minimum_nights",
        "min_nights_to_availability_ratio",
        "time_segment",
    ]
    drop_cols_sub = base_drop_cols + time_drop_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["review_scores_rating"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    xgb_seg = XGBRegressor(**xgb_params)
    xgb_seg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    pred_tr = xgb_seg.predict(X_train)
    pred_te = xgb_seg.predict(X_test)

    tr_r2 = r2_score(y_train, pred_tr)
    te_r2 = r2_score(y_test, pred_te)
    tr_mae = mean_absolute_error(y_train, pred_tr)
    te_mae = mean_absolute_error(y_test, pred_te)

    tr_adj_r2 = calculate_adjusted_r2(tr_r2, X_train.shape[0], X_train.shape[1])
    te_adj_r2 = calculate_adjusted_r2(te_r2, X_test.shape[0], X_test.shape[1])

    print(f"\n▶ SEGMENT: {seg_name} (N = {len(df_sub)})")
    print("-" * 60)
    print(f"Train R²:          {tr_r2:.4f}")
    print(f"Train Adjusted R²: {tr_adj_r2:.4f}")
    print(f"Train MAE:         {tr_mae:.4f}")
    print("-" * 60)
    print(
        f"Test R²:           {te_r2:.4f}  (Delta zu Global Baseline: {te_r2 - g_te_r2:+.4f})"
    )
    print(
        f"Test Adjusted R²:  {te_adj_r2:.4f}  (Delta zu Global Baseline: {te_adj_r2 - g_te_adj_r2:+.4f})"
    )
    print(
        f"Test MAE:          {te_mae:.4f}  (Delta zu Global Baseline: {te_mae - g_te_mae:+.4f})"
    )
    print("#" * 88)

--- STARTE FINALE EVALUIERUNG: OHE-NACHBARSCHAFTEN (INKL. ADJUSTIERTEM R²) ---

=== GLOBALE BASELINE ERGEBNISSE (GESAMTMARKT MIT OHE-KIEZEN) ===
Datensätze Gesamtmarkt (N): 6298
----------------------------------------------------------------------------------------
Global Train R²:           0.5133
Global Train Adjusted R²:  0.4912
Global Train MAE:          0.1228
----------------------------------------------------------------------------------------
Global Test R²:            0.2820
Global Test Adjusted R²:   0.1745
Global Test MAE:           0.1431


=== 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE) ===

▶ SEGMENT: Entire home/apt (N = 4546)
------------------------------------------------------------
Train R²:          0.5299
Train Adjusted R²: 0.5006
Train MAE:         0.1182
------------------------------------------------------------
Test R²:           0.3703  (Delta zu Global Baseline: +0.0883)
Test Adjusted R²:  0.2357  (Delta zu Global Baseline: +0.0613)
Test MAE:          0.

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


def calculate_adjusted_r2(r2, n, p):
    if n - p - 1 <= 0:
        return np.nan
    return 1 - ((1 - r2) * (n - 1) / (n - p - 1))


# 0. Daten laden
try:
    df_pipeline = df_model_target_encoded.copy()
except NameError:
    df_pipeline = pd.read_pickle("df_model_target_encoded.pkl")

# Bereinigung
base_drop_cols = ["review_scores_rating"]
for c in [
    "review_scores_location",
    "review_neighbourhood_rating",
    "neighbourhood_cleansed",
]:
    if c in df_pipeline.columns:
        base_drop_cols.append(c)

xgb_params = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
}

print(
    "--- STARTE FINALE EVALUIERUNG: TARGET-ENCODED NACHBARSCHAFTEN (INKL. ADJUSTIERTEM R²) ---\n"
)

# 1. GLOBALE BASELINE
y_global = df_pipeline["review_scores_rating"]
X_global = df_pipeline.drop(columns=base_drop_cols)

X_g_temp, X_g_test, y_g_temp, y_g_test = train_test_split(
    X_global, y_global, test_size=0.20, random_state=42
)
X_g_train, X_g_val, y_g_train, y_g_val = train_test_split(
    X_g_temp, y_g_temp, test_size=0.25, random_state=42
)

xgb_global = XGBRegressor(**xgb_params)
xgb_global.fit(X_g_train, y_g_train, eval_set=[(X_g_val, y_g_val)], verbose=False)

pred_g_train = xgb_global.predict(X_g_train)
pred_g_test = xgb_global.predict(X_g_test)

g_tr_r2 = r2_score(y_g_train, pred_g_train)
g_te_r2 = r2_score(y_g_test, pred_g_test)
g_tr_mae = mean_absolute_error(y_g_train, pred_g_train)
g_te_mae = mean_absolute_error(y_g_test, pred_g_test)

g_tr_adj_r2 = calculate_adjusted_r2(g_tr_r2, X_g_train.shape[0], X_g_train.shape[1])
g_te_adj_r2 = calculate_adjusted_r2(g_te_r2, X_g_test.shape[0], X_g_test.shape[1])

print("=" * 88)
print("=== GLOBALE BASELINE ERGEBNISSE (GESAMTMARKT MIT TARGET ENCODING) ===")
print(f"Datensätze Gesamtmarkt (N): {len(df_pipeline)}")
print("-" * 88)
print(f"Global Train R²:           {g_tr_r2:.4f}")
print(f"Global Train Adjusted R²:  {g_tr_adj_r2:.4f}")
print(f"Global Train MAE:          {g_tr_mae:.4f}")
print("-" * 88)
print(f"Global Test R²:            {g_te_r2:.4f}")
print(f"Global Test Adjusted R²:   {g_te_adj_r2:.4f}")
print(f"Global Test MAE:           {g_te_mae:.4f}")
print("=" * 88 + "\n\n")

# 2. SEGMENTIERUNG NACH ZIMMERTYP
print("=" * 88)
print("=== 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE) ===")
print("=" * 88)

room_dummy_cols = [c for c in df_pipeline.columns if c.startswith("room_type_")]

for col in room_dummy_cols:
    seg_name = col.replace("room_type_", "")
    df_sub = df_pipeline[df_pipeline[col] == 1].copy()

    if len(df_sub) < 150:
        continue

    drop_cols_sub = base_drop_cols + room_dummy_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["review_scores_rating"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    xgb_seg = XGBRegressor(**xgb_params)
    xgb_seg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    pred_tr = xgb_seg.predict(X_train)
    pred_te = xgb_seg.predict(X_test)

    tr_r2 = r2_score(y_train, pred_tr)
    te_r2 = r2_score(y_test, pred_te)
    tr_mae = mean_absolute_error(y_train, pred_tr)
    te_mae = mean_absolute_error(y_test, pred_te)

    tr_adj_r2 = calculate_adjusted_r2(tr_r2, X_train.shape[0], X_train.shape[1])
    te_adj_r2 = calculate_adjusted_r2(te_r2, X_test.shape[0], X_test.shape[1])

    print(f"\n▶ SEGMENT: {seg_name} (N = {len(df_sub)})")
    print("-" * 60)
    print(f"Train R²:          {tr_r2:.4f}")
    print(f"Train Adjusted R²: {tr_adj_r2:.4f}")
    print(f"Train MAE:         {tr_mae:.4f}")
    print("-" * 60)
    print(
        f"Test R²:           {te_r2:.4f}  (Delta zu Global Baseline: {te_r2 - g_te_r2:+.4f})"
    )
    print(
        f"Test Adjusted R²:  {te_adj_r2:.4f}  (Delta zu Global Baseline: {te_adj_r2 - g_te_adj_r2:+.4f})"
    )
    print(
        f"Test MAE:          {te_mae:.4f}  (Delta zu Global Baseline: {te_mae - g_te_mae:+.4f})"
    )
    print("#" * 88)

# 3. SEGMENTIERUNG NACH REISE-MOTIVATION
print("\n\n" + "=" * 88)
print("=== 3. SEGMENTIERUNG NACH REISE-MOTIVATION (ZEIT-HORIZONT) ===")
print("=" * 88)

df_pipeline["time_segment"] = np.where(
    df_pipeline["minimum_nights"] < 30,
    "1_KURZZEIT_TOURISMUS (< 30 Nächte)",
    "2_LANGZEIT_WOHNEN (>= 30 Nächte)",
)

zeit_segmente = sorted(df_pipeline["time_segment"].unique())

for seg_name in zeit_segmente:
    df_sub = df_pipeline[df_pipeline["time_segment"] == seg_name].copy()

    if len(df_sub) < 100:
        continue

    time_drop_cols = [
        "minimum_nights",
        "min_nights_to_availability_ratio",
        "time_segment",
    ]
    drop_cols_sub = base_drop_cols + time_drop_cols
    X_sub = df_sub.drop(columns=[c for c in drop_cols_sub if c in df_sub.columns])
    y_sub = df_sub["review_scores_rating"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_sub, y_sub, test_size=0.20, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    xgb_seg = XGBRegressor(**xgb_params)
    xgb_seg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    pred_tr = xgb_seg.predict(X_train)
    pred_te = xgb_seg.predict(X_test)

    tr_r2 = r2_score(y_train, pred_tr)
    te_r2 = r2_score(y_test, pred_te)
    tr_mae = mean_absolute_error(y_train, pred_tr)
    te_mae = mean_absolute_error(y_test, pred_te)

    tr_adj_r2 = calculate_adjusted_r2(tr_r2, X_train.shape[0], X_train.shape[1])
    te_adj_r2 = calculate_adjusted_r2(te_r2, X_test.shape[0], X_test.shape[1])

    print(f"\n▶ SEGMENT: {seg_name} (N = {len(df_sub)})")
    print("-" * 60)
    print(f"Train R²:          {tr_r2:.4f}")
    print(f"Train Adjusted R²: {tr_adj_r2:.4f}")
    print(f"Train MAE:         {tr_mae:.4f}")
    print("-" * 60)
    print(
        f"Test R²:           {te_r2:.4f}  (Delta zu Global Baseline: {te_r2 - g_te_r2:+.4f})"
    )
    print(
        f"Test Adjusted R²:  {te_adj_r2:.4f}  (Delta zu Global Baseline: {te_adj_r2 - g_te_adj_r2:+.4f})"
    )
    print(
        f"Test MAE:          {te_mae:.4f}  (Delta zu Global Baseline: {te_mae - g_te_mae:+.4f})"
    )
    print("#" * 88)

--- STARTE FINALE EVALUIERUNG: TARGET-ENCODED NACHBARSCHAFTEN (INKL. ADJUSTIERTEM R²) ---

=== GLOBALE BASELINE ERGEBNISSE (GESAMTMARKT MIT TARGET ENCODING) ===
Datensätze Gesamtmarkt (N): 6298
----------------------------------------------------------------------------------------
Global Train R²:           0.5183
Global Train Adjusted R²:  0.5145
Global Train MAE:          0.1209
----------------------------------------------------------------------------------------
Global Test R²:            0.2691
Global Test Adjusted R²:   0.2512
Global Test MAE:           0.1435


=== 2. SEGMENTIERUNG NACH ZIMMERTYP (ROOM TYPE) ===

▶ SEGMENT: Entire home/apt (N = 4546)
------------------------------------------------------------
Train R²:          0.5438
Train Adjusted R²: 0.5394
Train MAE:         0.1164
------------------------------------------------------------
Test R²:           0.3519  (Delta zu Global Baseline: +0.0828)
Test Adjusted R²:  0.3328  (Delta zu Global Baseline: +0.0816)
Test 